# Descarga de datos desde la NBA API

Este notebook descarga los *game logs* de equipo y de jugador para todas
las temporadas del intervalo 2006-07 a 2024-25 mediante el paquete
`nba_api`. Los ficheros CSV resultantes se guardan en `datasets/nba_api/`
y son la base sobre la que se construye la tabla `master` en
`dataset.ipynb`.

**Salidas generadas:**
- `team_game_logs.csv` — un registro por equipo y partido.
- `player_game_logs.csv` — un registro por jugador y partido.

In [ ]:
# === Setup ===
# Importaciones y configuración inicial. Se define el rango de temporadas
# a descargar y el directorio de salida.
import os, time
import numpy as np
import pandas as pd
from nba_api.stats.endpoints import LeagueGameLog

# Temporadas a descargar. Se cubre desde 2006-07 (primera con datos
# advanced disponibles) hasta la temporada más reciente completada.
SEASONS = ['2006-07','2007-08','2008-09','2009-10','2010-11','2011-12',
           '2012-13','2013-14','2014-15','2015-16','2016-17','2017-18',
           '2018-19','2019-20','2020-21','2021-22','2022-23','2023-24','2024-25']

OUT_DIR = 'datasets/nba_api'
os.makedirs(OUT_DIR, exist_ok=True)

## Descarga de los *game logs*

Se descargan dos vistas del endpoint `LeagueGameLog`, la de equipo (`'T'`)
y la de jugador (`'P'`). Cada llamada devuelve un DataFrame por temporada
que se concatena en un único frame. Entre peticiones se introduce una
pausa breve para no saturar la API oficial.

In [ ]:
# === Descarga de logs por equipo y por jugador ===
def pull_logs(player_or_team, label):
    """
    Descarga los game logs de todas las temporadas para la vista indicada.

    player_or_team : 'T' para logs por equipo, 'P' para logs por jugador.
    label          : etiqueta descriptiva usada solo para el print.
    """
    frames = []
    print(f'=== {label} ===')
    for s in SEASONS:
        try:
            df = LeagueGameLog(season=s,
                               season_type_all_star='Regular Season',
                               player_or_team_abbreviation=player_or_team,
                               timeout=30).get_data_frames()[0]
            df['SEASON'] = s
            frames.append(df)
            print(f'  {s}: {len(df):>6}')
            time.sleep(0.6)   # pausa entre peticiones para no saturar la API
        except Exception as e:
            print(f'  {s}: error -> {e}')
    out = pd.concat(frames, ignore_index=True)
    out['GAME_DATE'] = pd.to_datetime(out['GAME_DATE'])
    return out

team_logs   = pull_logs('T', 'TEAM LOGS')
player_logs = pull_logs('P', 'PLAYER LOGS')

print(f'\nTeam-game  rows: {len(team_logs):,}')
print(f'Player-game rows: {len(player_logs):,}')

=== TEAM LOGS ===
  2006-07:   2460
  2007-08:   2460
  2008-09:   2460
  2009-10:   2460
  2010-11:   2460
  2011-12:   1980
  2012-13:   2460
  2013-14:   2460
  2014-15:   2460
  2015-16:   2460
  2016-17:   2460
  2017-18:   2460
  2018-19:   2460
  2019-20:   2118
  2020-21:   2160
  2021-22:   2460
  2022-23:   2460
  2023-24:   2460
  2024-25:   2460
=== PLAYER LOGS ===
  2006-07:  25086
  2007-08:  24886
  2008-09:  24629
  2009-10:  24813
  2010-11:  25153
  2011-12:  20758
  2012-13:  25757
  2013-14:  25618
  2014-15:  25981
  2015-16:  26078
  2016-17:  26139
  2017-18:  26107
  2018-19:  26101
  2019-20:  22393
  2020-21:  23054
  2021-22:  26039
  2022-23:  25895
  2023-24:  26401
  2024-25:  26306

Team-game  rows: 45,618
Player-game rows: 477,194


## Guardado en disco

Los dos DataFrames se guardan como CSV en `datasets/nba_api/`. A partir
de aquí, el resto de notebooks solo necesitan leer estos ficheros, sin
volver a atacar la API.

In [3]:
team_logs.to_csv(os.path.join(OUT_DIR, 'team_game_logs.csv'), index=False)
player_logs.to_csv(os.path.join(OUT_DIR, 'player_game_logs.csv'), index=False)

## Verificación rápida de los datos descargados

Lectura de los CSV y muestra de las primeras filas de la tabla de equipos
para comprobar que la descarga se ha realizado correctamente.

In [4]:
team_df = pd.read_csv(os.path.join(OUT_DIR, 'team_game_logs.csv'))
player_df = pd.read_csv(os.path.join(OUT_DIR, 'player_game_logs.csv'))

pd.set_option('display.max_columns', None)
#player_df.head()
team_df.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,SEASON
0,22006,1610612756,PHX,Phoenix Suns,20600002,2006-10-31,PHX @ LAL,L,240,40,77,0.519,13,30,0.433,13,17,0.765,4,25,29,29,7,5,21,25,106,-8,0,2006-07
1,22006,1610612741,CHI,Chicago Bulls,20600001,2006-10-31,CHI @ MIA,W,240,39,79,0.494,7,13,0.538,23,32,0.719,13,36,49,22,10,5,16,21,108,42,0,2006-07
2,22006,1610612747,LAL,Los Angeles Lakers,20600002,2006-10-31,LAL vs. PHX,W,240,46,83,0.554,6,12,0.500,16,24,0.667,12,31,43,30,11,1,21,19,114,8,0,2006-07
3,22006,1610612748,MIA,Miami Heat,20600001,2006-10-31,MIA vs. CHI,L,240,25,65,0.385,3,17,0.176,13,22,0.591,4,25,29,13,6,3,23,24,66,-42,0,2006-07
4,22006,1610612744,GSW,Golden State Warriors,20600015,2006-11-01,GSW vs. LAL,L,240,34,82,0.415,3,19,0.158,27,42,0.643,12,34,46,18,6,4,10,31,98,-12,0,2006-07


## Utilidad: jugadores por partido

Función auxiliar que devuelve, para un `GAME_ID` dado, la lista de
jugadores que aparecieron en el registro, ordenados por puntos. Se usa
más adelante para validar los cruces con el parte de inactivos.

In [5]:
player_df = pd.read_csv(os.path.join(OUT_DIR, 'player_game_logs.csv'))

def get_game_players_by_id(game_id):
    return (
        player_df[player_df['GAME_ID'] == game_id]
        [['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION', 'PTS']]
        .sort_values(by=['TEAM_ABBREVIATION', 'PTS'], ascending=[True, False])
    )

get_game_players_by_id(20600001)

,PLAYER_ID,PLAYER_NAME,TEAM_ABBREVIATION,PTS
22,2550,Kirk Hinrich,CHI,26
35,2768,Chris Duhon,CHI,20
13,2736,Luol Deng,CHI,12
36,200757,Thabo Sefolosha,CHI,11
0,2804,Andres Nocioni,CHI,9
34,2732,Ben Gordon,CHI,6
10,1112,Ben Wallace,CHI,5
26,2751,Viktor Khryapa,CHI,5
6,200748,Tyrus Thomas,CHI,4
12,2124,Malik Allen,CHI,4


## Pruebas con otros endpoints de la API

Bloque exploratorio de los distintos endpoints ofrecidos por `nba_api`
(`BoxScoreTraditionalV2`, `BoxScoreAdvancedV3`, `BoxScoreFourFactorsV3`,
`TeamGameLogs`, `PlayerGameLogs`). Se dejan como referencia para futuras
extensiones del pipeline.

In [113]:
from nba_api.stats.endpoints import LeagueGameLog, BoxScoreTraditionalV2, BoxScoreSummaryV2, BoxScoreAdvancedV3, BoxScoreFourFactorsV3
pd.set_option('display.max_columns', None)
df_test = LeagueGameLog(season='2020-21', season_type_all_star='Regular Season', player_or_team_abbreviation='P').get_data_frames()[0]
#df_test = LeagueGameLog(season='2020-21', season_type_all_star='Regular Season', player_or_team_abbreviation='T').get_data_frames()[0]
df_test = BoxScoreTraditionalV2(game_id='0022300008').get_data_frames()[1]
#df_test = BoxScoreFourFactorsV3(game_id='0022300008').get_data_frames()[0]
#df_test = ScoreboardV2(team_id=1610612737).get_data_frames()[0]

df_test#.sort_values(by='GAME_ID', ascending=False)

from nba_api.stats.endpoints import teamgamelogs; df = teamgamelogs.TeamGameLogs(season_nullable="2024-25", season_type_nullable="Regular Season", measure_type_player_game_logs_nullable="Base").get_data_frames()[0]
#from nba_api.stats.endpoints import playergamelogs; df = playergamelogs.PlayerGameLogs(season_nullable="2024-25", season_type_nullable="Regular Season", measure_type_player_game_logs_nullable="Advanced").get_data_frames()[0]
df_test

C:\Users\ignac\AppData\Local\Temp\ipykernel_24712\3162397706.py:5: DeprecationWarning: BoxScoreTraditionalV2 is deprecated and will be removed in a future version. Please use BoxScoreTraditionalV3 instead. Data is no longer being published for BoxScoreTraditionalV2 as of the 2025-26 NBA season.
  df_test = BoxScoreTraditionalV2(game_id='0022300008').get_data_frames()[1]


,GAME_ID,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,TEAM_CITY,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS
0,0022300008,1610612755,76ers,PHI,Philadelphia,240:00,35,83,0.422,9,28,0.321,35,41,0.854,11,35,46,23,13,5,8,23,114,8.0
1,0022300008,1610612765,Pistons,DET,Detroit,240:00,40,86,0.465,8,28,0.286,18,22,0.818,11,37,48,28,3,5,16,26,106,-8.0


## Pruebas con otros endpoints de la API

Bloque exploratorio de los distintos endpoints ofrecidos por `nba_api`
(`BoxScoreTraditionalV2`, `BoxScoreAdvancedV3`, `BoxScoreFourFactorsV3`,
`TeamGameLogs`, `PlayerGameLogs`). Se dejan como referencia para futuras
extensiones del pipeline.

In [50]:
for i in range(9):
    print(f'=== DataFrame {i} ===')
    df = BoxScoreSummaryV2(game_id='0020600001').get_data_frames()[i]
    print(df)
    print('\n')

=== DataFrame 0 ===


C:\Users\ignac\AppData\Local\Temp\ipykernel_24712\3550324263.py:3: UserWarning: BoxScoreSummaryV2 has known data availability issues. Data may be missing for games on or after 4/10/2025. Users should moving to BoxScoreSummaryV3 or verify data completeness for their specific use cases and implement appropriate error handling.
  df = BoxScoreSummaryV2(game_id='0020600001').get_data_frames()[i]


         GAME_DATE_EST  GAME_SEQUENCE     GAME_ID  GAME_STATUS_ID GAME_STATUS_TEXT         GAMECODE  HOME_TEAM_ID  VISITOR_TEAM_ID SEASON  LIVE_PERIOD LIVE_PC_TIME NATL_TV_BROADCASTER_ABBREVIATION  \
0  2006-10-31T00:00:00              1  0020600001               3            Final  20061031/CHIMIA    1610612748       1610612741   2006            4                                           TNT   

  LIVE_PERIOD_TIME_BCAST  WH_STATUS  
0         Q4       - TNT          1  


=== DataFrame 1 ===
  LEAGUE_ID     TEAM_ID TEAM_ABBREVIATION TEAM_CITY  PTS_PAINT  PTS_2ND_CHANCE  PTS_FB  LARGEST_LEAD  LEAD_CHANGES  TIMES_TIED  TEAM_TURNOVERS  TOTAL_TURNOVERS  TEAM_REBOUNDS  PTS_OFF_TO
0        00  1610612741               CHI   Chicago         36              18      21            42             2           3               0               16             11          32
1        00  1610612748               MIA     Miami         26               2       6             2             2           3 

C:\Users\ignac\AppData\Local\Temp\ipykernel_24712\3550324263.py:3: UserWarning: BoxScoreSummaryV2 has known data availability issues. Data may be missing for games on or after 4/10/2025. Users should moving to BoxScoreSummaryV3 or verify data completeness for their specific use cases and implement appropriate error handling.
  df = BoxScoreSummaryV2(game_id='0020600001').get_data_frames()[i]
C:\Users\ignac\AppData\Local\Temp\ipykernel_24712\3550324263.py:3: UserWarning: BoxScoreSummaryV2 has known data availability issues. Data may be missing for games on or after 4/10/2025. Users should moving to BoxScoreSummaryV3 or verify data completeness for their specific use cases and implement appropriate error handling.
  df = BoxScoreSummaryV2(game_id='0020600001').get_data_frames()[i]


   OFFICIAL_ID FIRST_NAME   LAST_NAME JERSEY_NUM
0         1156        Tim     Donaghy       21  
1         1163     Bernie       Fryer       7   
2         1204      Derek  Richardson       63  


=== DataFrame 3 ===
   PLAYER_ID FIRST_NAME        LAST_NAME JERSEY_NUM     TEAM_ID TEAM_CITY TEAM_NAME TEAM_ABBREVIATION
0       2552    Michael         Sweetney       50    1610612741   Chicago     Bulls               CHI
1       2857      Andre          Barrett       11    1610612741   Chicago     Bulls               CHI
2     101149   Martynas  Andriuskevicius       15    1610612741   Chicago     Bulls               CHI
3       2853       Earl           Barron       30    1610612748     Miami      Heat               MIA
4       1715      Jason         Williams       55    1610612748     Miami      Heat               MIA
5       1720    Michael           Doleac       51    1610612748     Miami      Heat               MIA


=== DataFrame 4 ===


...

                   GAME_DATE  ATTENDANCE GAME_TIME
0  Tuesday, October 31, 2006       20229      2:26


=== DataFrame 5 ===
         GAME_DATE_EST  GAME_SEQUENCE     GAME_ID     TEAM_ID TEAM_ABBREVIATION TEAM_CITY_NAME TEAM_NICKNAME TEAM_WINS_LOSSES  PTS_QTR1  PTS_QTR2  PTS_QTR3  PTS_QTR4  PTS_OT1  PTS_OT2  PTS_OT3  \
0  2006-10-31T00:00:00              1  0020600001  1610612741               CHI        Chicago         Bulls              1-0        22        37        21        28        0        0        0   
1  2006-10-31T00:00:00              1  0020600001  1610612748               MIA          Miami          Heat              0-1        16        14        21        15        0        0        0   

   PTS_OT4 PTS_OT5 PTS_OT6 PTS_OT7 PTS_OT8 PTS_OT9 PTS_OT10  PTS  
0        0    None    None    None    None    None     None  108  
1        0    None    None    None    None    None     None   66  


=== DataFrame 6 ===


C:\Users\ignac\AppData\Local\Temp\ipykernel_24712\3550324263.py:3: UserWarning: BoxScoreSummaryV2 has known data availability issues. Data may be missing for games on or after 4/10/2025. Users should moving to BoxScoreSummaryV3 or verify data completeness for their specific use cases and implement appropriate error handling.
  df = BoxScoreSummaryV2(game_id='0020600001').get_data_frames()[i]
C:\Users\ignac\AppData\Local\Temp\ipykernel_24712\3550324263.py:3: UserWarning: BoxScoreSummaryV2 has known data availability issues. Data may be missing for games on or after 4/10/2025. Users should moving to BoxScoreSummaryV3 or verify data completeness for their specific use cases and implement appropriate error handling.
  df = BoxScoreSummaryV2(game_id='0020600001').get_data_frames()[i]


      GAME_ID LAST_GAME_ID   LAST_GAME_DATE_EST  LAST_GAME_HOME_TEAM_ID LAST_GAME_HOME_TEAM_CITY LAST_GAME_HOME_TEAM_NAME LAST_GAME_HOME_TEAM_ABBREVIATION  LAST_GAME_HOME_TEAM_POINTS  \
0  0020600001   0040500116  2006-05-04T00:00:00              1610612748                    Miami                     Heat                              MIA                         113   

   LAST_GAME_VISITOR_TEAM_ID LAST_GAME_VISITOR_TEAM_CITY LAST_GAME_VISITOR_TEAM_NAME LAST_GAME_VISITOR_TEAM_CITY1  LAST_GAME_VISITOR_TEAM_POINTS  
0                 1610612741                     Chicago                       Bulls                          CHI                             96  


=== DataFrame 7 ===
      GAME_ID  HOME_TEAM_ID  VISITOR_TEAM_ID        GAME_DATE_EST  HOME_TEAM_WINS  HOME_TEAM_LOSSES SERIES_LEADER
0  0020600001    1610612748       1610612741  2006-10-31T00:00:00               1                 3       Chicago


=== DataFrame 8 ===


C:\Users\ignac\AppData\Local\Temp\ipykernel_24712\3550324263.py:3: UserWarning: BoxScoreSummaryV2 has known data availability issues. Data may be missing for games on or after 4/10/2025. Users should moving to BoxScoreSummaryV3 or verify data completeness for their specific use cases and implement appropriate error handling.
  df = BoxScoreSummaryV2(game_id='0020600001').get_data_frames()[i]
C:\Users\ignac\AppData\Local\Temp\ipykernel_24712\3550324263.py:3: UserWarning: BoxScoreSummaryV2 has known data availability issues. Data may be missing for games on or after 4/10/2025. Users should moving to BoxScoreSummaryV3 or verify data completeness for their specific use cases and implement appropriate error handling.
  df = BoxScoreSummaryV2(game_id='0020600001').get_data_frames()[i]


      GAME_ID  VIDEO_AVAILABLE_FLAG  PT_AVAILABLE  PT_XYZ_AVAILABLE  WH_STATUS  HUSTLE_STATUS  HISTORICAL_STATUS
0  0020600001                     0             0                 0          1              1                  0


